### Model and Results

In [ ]:
# LIBS
import json
import re
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import warnings
# warnings.filterwarnings('ignore')
from sklearn.preprocessing import MinMaxScaler
from scipy import stats
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, layers, callbacks, models, optimizers
from tensorflow.keras.layers import Dense, LSTM, Dropout, GRU, Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
import seaborn as sns
from pandas.plotting import register_matplotlib_converters
# import statsmodels.api as sm
# from pmdarima import auto_arima
#bibliotecas usadas no gridserach
from sklearn.model_selection import TimeSeriesSplit
import keras
#salvar arquivo de treinamento
from tensorflow.keras.models import load_model


register_matplotlib_converters()
sns.set_style("darkgrid")

plt.rc("figure", figsize=(16, 6))
plt.rc("font", size=13)

from matplotlib.pyplot import figure

figure(figsize = (16, 6), dpi = 100)

In [ ]:
#gridsearch
def create_dataset (X, look_back):
    Xs, ys = [], []
    
    for i in range(len(X)-look_back):
        v = X[i:i+look_back]
        Xs.append(v)
        ys.append(X[i+look_back])
        
    return np.array(Xs), np.array(ys)

def grid_search_cv(modelo, units, X_train, y_train,learning_rates, epochs_list, batch_sizes, patiences, model_name):
    best_loss = float('inf')
    best_params = {}
    best_model = None
    best_history = None
    look_back = 0
    #for look_back in lb_list:
    for lr in learning_rates:
        for epochs in epochs_list:
            for batch_size in batch_sizes:
                for patience in patiences:
                    #X_train, y_train = create_dataset(train_scaled, look_back)
                    #X_test, y_test = create_dataset(test_scaled, look_back)
                    model = modelo(units, X_train, lr)
                    histories = fit_model_with_cross_validation(model, X_train, y_train, model_name, patience, epochs, batch_size)
                    mean_history = calculate_mean_history(histories)
                    val_loss = min(mean_history['val_loss'])
                    print("Val Loss: ", val_loss, "learning rate: ", lr, "epochs: ",  epochs, "batch_size: " , batch_size, "patience: ", patience)
                    if val_loss < best_loss:
                        best_loss = val_loss
                        best_params = {'learning_rate': lr, 'epochs': epochs, 'batch_size': batch_size, 'patience': patience}
                        best_model = model  
                        best_history = mean_history  
    print(f'O modelo {model_name} tem como melhores parâmetros: learning_rate {best_params["learning_rate"]}, epochs {best_params["epochs"]}, batch_size {best_params["batch_size"]}, patience {best_params["patience"]}')
    return best_model, best_history, best_params


#validação cruzada
def fit_model_with_cross_validation(model, xtrain, ytrain, model_name, patience, epochs, batch_size):
    tscv = TimeSeriesSplit(n_splits=5)
    fold = 1
    histories = []
    for train_index, val_index in tscv.split(xtrain):
        x_train_fold, x_val_fold = xtrain[train_index], xtrain[val_index]
        y_train_fold, y_val_fold = ytrain[train_index], ytrain[val_index]
        early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True, min_delta=1e-5)
        history = model.fit(x_train_fold, y_train_fold, epochs=epochs, validation_data=(x_val_fold, y_val_fold), batch_size=batch_size, callbacks=[early_stop], verbose=1)
        print('\n\nTREINAMENTO - Fold', fold, 'do modelo:', model_name)
        histories.append(history)
        fold += 1   
    return histories 

# calcula a media das metricas obtidas nos historys - validação cruzada
def calculate_mean_history(histories):
    mean_history = {'loss': [], 'root_mean_squared_error': [], 'val_loss': [], 'val_root_mean_squared_error': []}
    for fold_history in histories:
        for key in mean_history.keys():
            mean_history[key].append(fold_history.history[key])
    for key, values in mean_history.items():
        max_len = max(len(val) for val in values)
        for i in range(len(values)):
            if len(values[i]) < max_len: #caso em que nao se treina todas as epocas (patience)
                values[i] += [values[i][-1]] * (max_len - len(values[i])) #completa o restante da lista com o ultimo valor obtido
    for key, values in mean_history.items():
        mean_history[key] = [sum(vals) / len(vals) for vals in zip(*values)]
    
    return mean_history

#função para salvar o modelo
def save_model(model, directory, substring_desejada, modelo):
    if not os.path.exists(directory):
        os.makedirs(directory)
    file_path = os.path.join(directory, f'{substring_desejada +modelo} - final_model.keras')
    model.save(file_path)
    print(f"Modelo salvo como '{file_path}'")

#plotar os graficos da media dos treinamentos por epocas: validação cruzada
def plot_loss_cv(mean_history, model_name, link):
    epochs = range(1, len(mean_history['loss']) + 1)
    plt.plot(epochs, mean_history['loss'], label='Train Loss')
    plt.plot(epochs, mean_history['val_loss'], label='Validation Loss')
    plt.title('Mean Training and Validation Loss for '+' '+link + ' '+ model_name)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

def plot_rmse_cv(mean_history):
    epochs = range(1, len(mean_history['root_mean_squared_error']) + 1)
    plt.plot(epochs, mean_history['root_mean_squared_error'], label='Train RMSE')
    plt.plot(epochs, mean_history['val_root_mean_squared_error'], label='Validation RMSE')
    plt.title('Mean Training and Validation RMSE')
    plt.xlabel('Epoch')
    plt.ylabel('RMSE')
    plt.legend()
    plt.show()


In [ ]:
# Main functions

# Create input dataset
# The input shape should be [samples, time steps, features

# Create GRU model
def create_gru(units, train, learning_rate): 
    model = Sequential() 
    # Old Config
    model.add(GRU(units = units, return_sequences = True, input_shape = [train.shape[1], train.shape[2]]))
    model.add(GRU(units = units))   
    # model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(loss=MeanSquaredError(), optimizer = Adam(learning_rate=learning_rate), metrics=[RootMeanSquaredError()])
    return model

# Create LSTM model
def create_lstm(units, train, learning_rate): 
    model = Sequential() 
    # Old Config
    model.add(LSTM(units = units, return_sequences = True, input_shape = [train.shape[1], train.shape[2]]))
    model.add(LSTM(units = units)) 
    # model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(loss=MeanSquaredError(), optimizer = Adam(learning_rate=learning_rate), metrics=[RootMeanSquaredError()])
    
    return model

#treinamento do modelo
def fit_model(model, xtrain, ytrain, model_name, patience, epochs, batch_size ):
    early_stop = keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = patience, restore_best_weights=True)
    history = model.fit(xtrain, ytrain, epochs = epochs, validation_split = 0.2, batch_size = batch_size, shuffle = True, callbacks=[early_stop]) 
    print('\n\nTREINAMENTO: ' + model_name)
    return history

########################################### plote dos graficos de treinamento ###################################################################################
 #Plot train loss and validation loss
def plot_loss(history, model_name, link):
     plt.figure(figsize = (15, 6), dpi=100)
     plt.plot(history.history['loss'])
     plt.plot(history.history['val_loss'])
     plt.title('Model Train vs Validation Loss for '+' '+link + ' '+ model_name)
     plt.ylabel('Loss')
     plt.xlabel('Epoch')
     plt.legend(['Train loss', 'Validation loss'], loc='upper right')
def plot_rmse(history, model_name, link):
     plt.figure(figsize = (15, 6), dpi=100)
     plt.plot(history.history['rmse'])
     plt.plot(history.history['val_rmse'])
     plt.title('Model Train vs RMSE for '+' '+link + ' '+ model_name)
     plt.ylabel('rmse')
     plt.xlabel('Epoch')
     plt.legend(['Train rmse', 'Validation loss'], loc='upper right')
################################################################################################################################################################
# Make prediction
def prediction(model, xtest, ytest, myscaler, model_name, link): 
    prediction = model.predict(xtest) 
    prediction = myscaler.inverse_transform(prediction) 
    # dataframe_prediction = pd.DataFrame(data={'Predições':prediction.flatten()})
    dataframe_prediction = pd.DataFrame(data={'Prediction':prediction.flatten(), 'Test':ytest.flatten()})
    #save_path = os.path.join('..', '..', 'predicoes', f'prediction {model_name} {link}.csv') 
    save_path = os.path.join('..', '..', 'results','window', 'forecast', f'prediction {model_name} {link}.csv') 
    dataframe_prediction.to_csv(save_path)
    return prediction

# Plot test data vs prediction
# def plot_future(predictionGRU, predictionLSTM, y_test, link):
#     plt.figure(figsize=(15, 6), dpi=100)
#     range_future = len(y_test)
#     plt.plot(np.arange(range_future), np.array(y_test), label='Test data')
#     plt.plot(np.arange(range_future), np.array(predictionGRU), label='GRU')
#     plt.plot(np.arange(range_future), np.array(predictionLSTM), label='LSTM')
#     # dict_to_dataframe_prediction = {
#     #     # "range_future": np.arange(range_future),
#     #     f"prediction{model_name}": np.array(prediction.squeeze())
#     # }
    
#     plt.title('Test data vs prediction for '+ link)
#     plt.legend(loc='upper left')
#     plt.xlabel('Time')
#     plt.ylabel('Mbis/s')

#     #Tenta salvar a fig
#     save_path = '../../graficos/predicoes/round_2/graficos/' + link + '.png'
#     try:
#         plt.savefig(save_path)
#         print(f"A figura foi salva com sucesso em: {save_path}")
#     except Exception as e:
#         print(f"Erro ao salvar a figura: {e}")

#     plt.show()

def plot_future(predictionGRU, predictionLSTM, predictionbiLSTM, predictionbiGRU, y_test, link):
    plt.figure(figsize=(15, 6), dpi=100)
    range_future = len(y_test)
    plt.plot(np.arange(range_future), np.array(y_test), label='Test data')
    plt.plot(np.arange(range_future), np.array(predictionbiLSTM), label='bi-LSTM')
    plt.plot(np.arange(range_future), np.array(predictionGRU), label='GRU')
    plt.plot(np.arange(range_future), np.array(predictionLSTM), label='LSTM')
    plt.plot(np.arange(range_future), np.array(predictionbiGRU), label='bi-GRU')
    # dict_to_dataframe_prediction = {
    #     # "range_future": np.arange(range_future),
    #     f"prediction{model_name}": np.array(prediction.squeeze())
    # }
    
    plt.title('Test data vs prediction for '+ link)
    plt.legend(loc='upper left')
    plt.xlabel('Time')
    plt.ylabel('Mbis/s')
    save_path = os.path.join('..', '..', 'results', 'window', 'plosts', link + '.png')
    save_path = os.path.normpath(save_path)  

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    try:
        plt.savefig(save_path)
        print(f"A figura foi salva com sucesso em: {save_path}")
    except Exception as e:
        print(f"Erro ao salvar a figura: {e}")
    plt.show()

    # #Tenta salvar a fig
    # save_path = os.path.join('..', '..', 'graficos', 'predicoes', 'round_2', 'graficos', link + '.png')

    # #save_path = '../../graficos/predicoes/round_2/graficos/' + link + '.png'
    # try:
    #     plt.savefig(save_path)
    #     print(f"A figura foi salva com sucesso em: {save_path}")
    # except Exception as e:
    #     print(f"Erro ao salvar a figura: {e}")

    # plt.show()
    
# Calculate MAE and RMSE
def evaluate_prediction(predictions, actual, model_name):
    errors = predictions - actual
    mse = np.square(errors).mean()
    rmse = np.sqrt(mse)
    nrmse = rmse/ ((np.max(actual))-(np.min(actual)))
    mae = np.abs(errors).mean()
    print(model_name + ':')
    print('Mean Absolute Error: {:.4f}'.format(mae))
    print('Root Mean Square Error: {:.4f}'.format(rmse))
    print('Normalized Root Mean Square Error: {:.4f}%'.format(nrmse*100))
    print('')

    return rmse, mae, nrmse, model_name

def bits_para_megabits(df, col_vaz):
    # Dados em MegaBits/s e fill
    df[col_vaz] = df[col_vaz]/1000000
    df[col_vaz] = df[col_vaz].replace(-1, df[col_vaz].mean())
    df[col_vaz] = df[col_vaz].fillna(df[col_vaz].mean())

    return df
                
def visualizacao_series(df, col_vazao, titulo):
    df[col_vazao].plot(figsize=(18,6))
    plt.title(titulo)
    plt.ylabel('Vazao (Mbits/s)')
    plt.legend() 
    plt.show()
#calcular media dia repetido
def calcular_media_dia(dataframe, col_data, col_vazao):
    # Ordena o DataFrame pelo campo 'Data'
    dataframe = dataframe.sort_values(by=[col_data])
    
    # Inicializa variáveis para rastrear o dia atual e os valores para calcular a média
    dia_atual = None
    valores_para_media = []
    
    # Função para calcular a média e substituir os valores '-1'
    def calcular_media_e_substituir():
        if len(valores_para_media) > 0:
            media = np.mean(valores_para_media)
            for idx in indices_para_substituir:
                dataframe.at[idx, col_vazao] = media
    
    indices_para_substituir = []
    
    # Itera pelas linhas do DataFrame
    for idx, row in dataframe.iterrows():
        data = row[col_data]
        vazao = row[col_vazao]
        
        if dia_atual is None:
            dia_atual = data
            valores_para_media = []
            indices_para_substituir = []
        
        # Verifica se o valor de 'Vazao' é diferente de '-1'
        if vazao != -1:
            valores_para_media.append(vazao)
        else:
            indices_para_substituir.append(idx)
        
        # Verifica se o dia mudou
        if data != dia_atual:
            calcular_media_e_substituir()
            dia_atual = data
            valores_para_media = []
            indices_para_substituir = []
    
    # Calcula a média e substitui os valores para o último dia
    calcular_media_e_substituir()
    
    return dataframe

# def plot_arima(pred, test, col_test, col_arima):

#     plt.plot(test[col_test], label= 'Real')
#     plt.plot(pred[col_arima], label='ARIMA')
#     plt.legend()
#     plt.xlabel('Tempo')
#     plt.ylabel('Vazão (bits/s)')
#     plt.title('Predição ARIMA')

# def create_arima(ts, col_vazao):

#     tamanho = int(len(ts.index) * 0.8)

#     # Split train data and test data
#     train_size = tamanho

#     # train_data = df.WC.loc[:train_size] -----> it gives a series
#     # Do not forget use iloc to select a number of rows
#     train_data = ts[:train_size]
#     test_data = ts[train_size:]

#     modelo_arima = auto_arima(train_data[col_vazao], seasonal=True, stepwise=True)

#     p, d, q = modelo_arima.order

#     modelo_manual = sm.tsa.ARIMA(train_data[col_vazao], order=(p, d, q)).fit()

#     previsoes = modelo_manual.predict(start=1, end=142, dynamic=False)

#     # Ajustar indice
#     test_data = test_data.reset_index(drop=True)
#     previsoes = pd.DataFrame(previsoes.reset_index(drop=True))

#    return previsoes, test_data

#Funções que ainda não estão acabada e não estão sendo utilizadas #############################################################################################
# def create_cnn(units, train):

#     model = Sequential()

#     model.add(Conv1D(units = units, kernel_sieze = 2, return_sequences=True, input_shape = [train.shape[1], train.shape[2]]))

#     model.add(Flatten())

#     model.add(Conv1D(units = units))

#     model.add(Dense(1))

#     model.compile(loss=MeanSquaredError(), optimizer = Adam(learning_rate=0.1), metrics=[RootMeanSquaredError()])

#     return model

In [ ]:
#TCN
def build_tcn(units, train, learning_rate):
    model = models.Sequential()
    model.add(layers.Conv1D(units, 3, activation='relu', padding='causal', dilation_rate=1, input_shape=[train.shape[1], train.shape[2]]))
    model.add(layers.Dropout(0.2))
    model.add(layers.Conv1D(units, 3, activation='relu', padding='causal', dilation_rate=2))
    model.add(layers.Dropout(0.2))
    model.add(layers.MaxPooling1D(pool_size=2))
    model.add(layers.Flatten())
    model.add(layers.Dense(units, activation='relu'))
    model.add(layers.Dropout(0.2))
    model.add(layers.Dense(1)) 
    #optimizer = optimizers.Adam(learning_rate=learning_rate)
    #model.compile(optimizer=optimizer, loss='mse')  # Para regressão. Para classificação, use 'categorical_crossentropy'.
    model.compile(loss = MeanSquaredError(), optimizer = Adam(learning_rate=learning_rate), metrics = [RootMeanSquaredError()])
    return model

def create_bi_lstm(units, train, learning_rate):
    model = Sequential()
    model.add(Bidirectional(LSTM(units=units, return_sequences=True), input_shape=[train.shape[1], train.shape[2]]))
    model.add(Bidirectional(LSTM(units=units)))
    model.add(Dense(1))
    model.compile(loss=MeanSquaredError(), optimizer=Adam(learning_rate=learning_rate), metrics=[RootMeanSquaredError()])

    return model

def create_bi_gru(units, train, learning_rate):
    model = Sequential()
    model.add(Bidirectional(GRU(units=units, return_sequences=True), input_shape=[train.shape[1], train.shape[2]]))
    model.add(Bidirectional(GRU(units=units)))
    model.add(Dense(1))
    model.compile(
        loss=MeanSquaredError(), 
        optimizer=Adam(learning_rate=learning_rate), 
        metrics=[RootMeanSquaredError()]
    )
    return model

In [ ]:
#salvando as saidas em um arquivo externo para analise 
import sys

# Redirecionar saída padrão para um arquivo com codificação utf-8
orig_stdout = sys.stdout
# f = open('training_output.txt', 'w', encoding='utf-8')
# sys.stdout = f

# Model Trainnig and Prediction
import os
import re
#treinamento e predição
# Defina o diretório raiz onde deseja iniciar a busca
diretorio_raiz = os.path.join('..', '..', 'datasets', 'imputed-choosen-best-svd')

evaluation = {}

# Itere pelos diretórios e subdiretórios
for pasta_raiz, subpastas, arquivos in os.walk(diretorio_raiz):
    for arquivo in arquivos:
        # Verifique se o arquivo é um arquivo CSV
        if arquivo.endswith('.csv'):
            # Construa o caminho completo para o arquivo
            caminho_arquivo = os.path.join(pasta_raiz, arquivo)
            # print(caminho_arquivo)

        try:
                # Título parser
                # partes = caminho_arquivo.split("\\")
                # if len(partes) >= 2:
                # #     substring_desejada = "/".join(partes[1:])  # Acesse as partes a partir da segunda e as una com barras invertidas
                partes = caminho_arquivo.split(os.sep)
                #partes = caminho_arquivo.split("/")
                substring_desejada = partes[4] + ' - ' + partes[5]  # Acesse as partes a partir da segunda e as una com barras invertidas
                
                #df = pd.read_csv(caminho_arquivo, index_col='Data')
                df = pd.read_csv(caminho_arquivo, index_col='Timestamp')
                if '0' in df.columns:
                    df = df.drop('0', axis=1)
                
                # Regularizar o dataset para megabits/s
                bits_para_megabits(df, 'Throughput')

                # Visualização das séries
                # visualizacao_series(df, 'Vazao', substring_desejada)

                # print('média dataset: '+ str(df['Vazao'].mean()))
                
                # print(substring_desejada)

                print('###################### '+substring_desejada+' ##########################')

                #PREDICAO
                tamanho = int(len(df.index) * 0.8)

                # Set random seed to get the same result after each time running the code
                tf.random.set_seed(7)

                # Split train data and test data
                train_size = tamanho

                # train_data = df.WC.loc[:train_size] -----> it gives a series
                # Do not forget use iloc to select a number of rows
                train_data = df[:train_size]
                test_data = df[train_size:]
                
                # Criar uma instância do MinMaxScaler
                train_data = train_data['Throughput'].values.reshape(-1, 1)
                test_data = test_data['Throughput'].values.reshape(-1, 1)

                scaler = MinMaxScaler().fit(train_data)

                train_scaled = scaler.transform(train_data)
                test_scaled = scaler.transform(test_data)

                X_train, y_train = create_dataset(train_scaled, 28)
                X_test, y_test = create_dataset(test_scaled, 28)

                list_lr = [0.0001] 
                list_epochs = [100, 200, 300] 
                list_bs = [32, 64, 128]
                list_pat = [5]
                lb_list = [10]

                model_bilstm, history_bilstm, best_params_bilstm = grid_search_cv(create_bi_lstm, 64, X_train, y_train,  list_lr,list_epochs, list_bs, list_pat, 'bi-lstm')
                model_gru, history_gru, best_params_gru = grid_search_cv(create_gru, 64, X_train, y_train, list_lr,list_epochs, list_bs, list_pat, 'gru')
                model_lstm, history_lstm, best_params_lstm = grid_search_cv(create_lstm, 64, X_train, y_train, list_lr, list_epochs, list_bs, list_pat, 'lstm')
                model_bigru, history_bigru, best_params_bigru = grid_search_cv(create_bi_gru, 64, X_train, y_train, list_lr, list_epochs, list_bs, list_pat, 'bi-gru')

                y_test = scaler.inverse_transform(y_test)
                y_train = scaler.inverse_transform(y_train)

                prediction_bilstm = prediction(model_bilstm, X_test, y_test, scaler, 'bi-LSTM', link = substring_desejada)
                prediction_gru = prediction(model_gru, X_test, y_test, scaler, 'GRU', link = substring_desejada)
                prediction_lstm = prediction(model_lstm, X_test, y_test, scaler, 'LSTM', link=substring_desejada)
                prediction_bigru = prediction(model_bigru, X_test, y_test, scaler, 'bi=GRU', link = substring_desejada)
                
                plot_loss_cv(history_bilstm, 'bi-LSTM', substring_desejada)
                plot_loss_cv(history_gru, 'GRU',substring_desejada )
                plot_loss_cv(history_lstm, 'LSTM', substring_desejada)
                plot_loss_cv(history_bigru, 'bi-GRU',substring_desejada )
                

                plot_future(prediction_gru, prediction_lstm, prediction_bilstm, prediction_bigru, y_test, link = substring_desejada)

                bilstm_evaluation = evaluate_prediction(prediction_bilstm, y_test, 'bi-LSTM')
                gru_evaluation = evaluate_prediction(prediction_gru, y_test, 'GRU')
                lstm_evaluation = evaluate_prediction(prediction_lstm, y_test, 'LSTM')
                bigru_evaluation = evaluate_prediction(prediction_bigru, y_test, 'bi-GRU')

                chave_bilstm = f"{substring_desejada}, {bilstm_evaluation[3]}"  
                tupla = (bilstm_evaluation[0], bilstm_evaluation[1], bilstm_evaluation[2])  
                evaluation[chave_bilstm] = tupla 

                chave_gru = f"{substring_desejada}, {gru_evaluation[3]}"  
                tupla = (gru_evaluation[0], gru_evaluation[1], gru_evaluation[2])  
                evaluation[chave_gru] = tupla  

                chave_lstm = f"{substring_desejada}, {lstm_evaluation[3]}" 
                tupla = (lstm_evaluation[0], lstm_evaluation[1], lstm_evaluation[2])  
                evaluation[chave_lstm] = tupla 
                
                chave_bigru = f"{substring_desejada}, {bigru_evaluation[3]}"  
                tupla = (bigru_evaluation[0], bigru_evaluation[1], bigru_evaluation[2])  
                evaluation[chave_bigru] = tupla
                    
        except pd.errors.EmptyDataError:
            print(f"Arquivo: {arquivo} - O arquivo está vazio, subpasta: {pasta_raiz}")
        except Exception as e:
                print(f"Arquivo: {arquivo}, subpasta: {pasta_raiz} - Erro: {str(e)}")

        novo_dicionario = {}
        
        for chave, valores in evaluation.items():
            valor1, valor2, valor3 = valores  # Desempacote os valores da tupla
            novo_dicionario[chave] = {'RMSE': valor1, 'MAE': valor2, 'NRMSE': valor3}

        json_path = os.path.join('..', '..', 'results', 'window', 'evaluation_rmse_mae_2.json')
        with open(json_path, 'w') as arquivo:
            json.dump(novo_dicionario, arquivo, indent=4)


In [ ]:
#salva os dados em um arquivo csv - esse sera o padrao que vamos manter

import json
import pandas as pd
caminho_arquivo = "../../graficos/predicoes/round_2/evaluation_rmse_mae_2.json"
arquivo_saida = "../../graficos/predicoes/round_2/evaluation_rmse_mae.csv"

with open(caminho_arquivo, 'r') as arquivo:
    data = json.load(arquivo)
csv_data = []

for key, metrics in data.items():
    parts = key.split()
    imputacao = parts[0]
    tcp = parts[3]
    link = parts[6]
    modelo = parts[-1].replace(",", "")  
    csv_data.append({
        "imputacao": imputacao,
        "link": link,
        "tcp": tcp,
        "modelo": modelo,
        "RMSE": metrics["RMSE"],
        "MAE": metrics["MAE"], 
        "NRMSE":metrics["NRMSE"]
    })

df = pd.DataFrame(csv_data)

df.to_csv(arquivo_saida, index=False)

In [ ]:
# Results PA-BA RMSE
# resultados referente a: '../../graficos/predicoes/round_2/evaluation_rmse_mae_2.json' que é
# resultado do treinamento com os hyperparametros em indicados em: '../../graficos/predicoes/round_2/hyperparams_2.txt'

import json
import matplotlib.pyplot as plt
import numpy as np


json_data = """
{
    "Interpolação Linear": [
      {"RMSE": 49.1656595492494, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 50.895106296664906, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 52.89093576345165, "MODELO": "GRU", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 66.27586771760345, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 163.7536402553969, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 164.4494189110069, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 153.7524221737528, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PR-AM"},
      {"RMSE": 162.36095430547388, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PR-AM"}
    ],
    "KNN": [
      {"RMSE": 58.27211696536066, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 61.56397645649821, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 48.2686337382006, "MODELO": "GRU", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 50.11289780679673, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 170.99390286197323, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 171.44232686161416, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 134.13692806013054, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PR-AM"},
      {"RMSE": 137.62774892075163, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PR-AM"}
    ],
    "Média Móvel": [
      {"RMSE": 262.62576432998407, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 266.2307921843741, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 272.7989797854411, "MODELO": "GRU", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 277.60481655611756, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 262.497759104749, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 266.94066414611785, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 273.2646414573494, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PR-AM"},
      {"RMSE": 279.2394819182783, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PR-AM"}
    ],
    "Mediana Móvel": [
      {"RMSE": 55.569133774622046, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 57.632766506417205, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 48.40798349140053, "MODELO": "GRU", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 64.51374974475104, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 251.11487261984317, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 251.0682412804039, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 141.88601538905303, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PR-AM"},
      {"RMSE": 144.71893528301663, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PR-AM"}
    ]
}
"""

# Carregar o JSON
data = json.loads(json_data)

# Dicionário para armazenar os valores de RMSE por grupo e configuração
rmse_values = {
    'Interpolação Linear': {'GRU_BBR_PA-BA': [], 'LSTM_CUBIC_PA-BA': [], 'LSTM_BBR_PA-BA': [], 'GRU_CUBIC_PA-BA': []},
    'KNN': {'GRU_BBR_PA-BA': [], 'LSTM_CUBIC_PA-BA': [], 'LSTM_BBR_PA-BA': [], 'GRU_CUBIC_PA-BA': []},
    'Média Móvel': {'GRU_BBR_PA-BA': [], 'LSTM_CUBIC_PA-BA': [], 'LSTM_BBR_PA-BA': [], 'GRU_CUBIC_PA-BA': []},
    'Mediana Móvel': {'GRU_BBR_PA-BA': [], 'LSTM_CUBIC_PA-BA': [], 'LSTM_BBR_PA-BA': [], 'GRU_CUBIC_PA-BA': []}
}

# Organizar os valores de RMSE no dicionário
for group, values in data.items():
    for item in values:
        rmse = item['RMSE']
        modelo = item['MODELO']
        tcp = item['TCP']
        link = item['LINK']
        
        if modelo == 'GRU' and tcp == 'BBR' and link == 'PA-BA':
            rmse_values[group]['GRU_BBR_PA-BA'].append(rmse)
        elif modelo == 'LSTM' and tcp == 'CUBIC' and link == 'PA-BA':
            rmse_values[group]['LSTM_CUBIC_PA-BA'].append(rmse)
        elif modelo == 'LSTM' and tcp == 'BBR' and link == 'PA-BA':
            rmse_values[group]['LSTM_BBR_PA-BA'].append(rmse)
        elif modelo == 'GRU' and tcp == 'CUBIC' and link == 'PA-BA':
            rmse_values[group]['GRU_CUBIC_PA-BA'].append(rmse)

# Criar o gráfico de barras
labels = list(rmse_values.keys())
x = np.arange(len(labels))
width = 0.08

fig, ax = plt.subplots()

# Adicionar as barras
bars1 = ax.bar(x - width, [np.mean(rmse_values[group]['GRU_BBR_PA-BA']) for group in labels], width, label='GRU - BBR', color='darkturquoise')
bars2 = ax.bar(x, [np.mean(rmse_values[group]['GRU_CUBIC_PA-BA']) for group in labels], width, label='GRU - CUBIC', color='blue')
bars3 = ax.bar(x + width, [np.mean(rmse_values[group]['LSTM_BBR_PA-BA']) for group in labels], width, label='LSTM - BBR', color='gold')
bars4 = ax.bar(x + 2*width, [np.mean(rmse_values[group]['LSTM_CUBIC_PA-BA']) for group in labels], width, label='LSTM - CUBIC', color='orange')

# Configurações do gráfico
ax.set_xlabel('Técnicas de Imputação', size= 24)
ax.set_ylabel('RMSE', size = 22)
# ax.set_title('RMSE - Ponto de Comunicação PA-BA', size = 24)
ax.set_xticks(x)
ax.set_xticklabels(labels, size = 20)
ax.legend()
ax.legend(fontsize = 17)


# plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()

In [ ]:
# Results PA-BA RMSE
# resultados referente a: '../../graficos/predicoes/round_2/evaluation_rmse_mae_2.json' que eh
# resultado do treinamento com os hyperparametros em indicados em: '../../graficos/predicoes/round_2/hyperparams_2.txt'

import json
import matplotlib.pyplot as plt
import numpy as np


json_data = """
{
    "Interpolação Linear": [
      {"RMSE": 49.1656595492494, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 50.895106296664906, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 52.89093576345165, "MODELO": "GRU", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 66.27586771760345, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 163.7536402553969, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 164.4494189110069, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 153.7524221737528, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PR-AM"},
      {"RMSE": 162.36095430547388, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PR-AM"}
    ],
    "KNN": [
      {"RMSE": 58.27211696536066, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 61.56397645649821, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 48.2686337382006, "MODELO": "GRU", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 50.11289780679673, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 170.99390286197323, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 171.44232686161416, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 134.13692806013054, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PR-AM"},
      {"RMSE": 137.62774892075163, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PR-AM"}
    ],
    "Média Móvel": [
      {"RMSE": 262.62576432998407, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 266.2307921843741, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 272.7989797854411, "MODELO": "GRU", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 277.60481655611756, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 262.497759104749, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 266.94066414611785, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 273.2646414573494, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PR-AM"},
      {"RMSE": 279.2394819182783, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PR-AM"}
    ],
    "Mediana Móvel": [
      {"RMSE": 55.569133774622046, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 57.632766506417205, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PA-BA"},
      {"RMSE": 48.40798349140053, "MODELO": "GRU", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 64.51374974475104, "MODELO": "LSTM", "TCP": "BBR", "LINK": "PR-AM"},
      {"RMSE": 251.11487261984317, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 251.0682412804039, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PA-BA"},
      {"RMSE": 141.88601538905303, "MODELO": "GRU", "TCP": "CUBIC", "LINK": "PR-AM"},
      {"RMSE": 144.71893528301663, "MODELO": "LSTM", "TCP": "CUBIC", "LINK": "PR-AM"}
    ]
}
"""

# Carregar o JSON
data = json.loads(json_data)

# Dicionário para armazenar os valores de RMSE por grupo e configuração
rmse_values = {
    'Interpolação Linear': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'KNN': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'Média Móvel': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'Mediana Móvel': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []}
}

# Organizar os valores de RMSE no dicionário
for group, values in data.items():
    for item in values:
        rmse = item['RMSE']
        modelo = item['MODELO']
        tcp = item['TCP']
        link = item['LINK']
        
        if modelo == 'GRU' and tcp == 'BBR' and link == 'PR-AM':
            rmse_values[group]['GRU_BBR_PR-AM'].append(rmse)
        elif modelo == 'LSTM' and tcp == 'CUBIC' and link == 'PR-AM':
            rmse_values[group]['LSTM_CUBIC_PR-AM'].append(rmse)
        elif modelo == 'LSTM' and tcp == 'BBR' and link == 'PR-AM':
            rmse_values[group]['LSTM_BBR_PR-AM'].append(rmse)
        elif modelo == 'GRU' and tcp == 'CUBIC' and link == 'PR-AM':
            rmse_values[group]['GRU_CUBIC_PR-AM'].append(rmse)

# Criar o gráfico de barras
labels = list(rmse_values.keys())
x = np.arange(len(labels))
width = 0.08

fig, ax = plt.subplots()

# Adicionar as barras
bars1 = ax.bar(x - width, [np.mean(rmse_values[group]['GRU_BBR_PR-AM']) for group in labels], width, label='GRU - BBR', color='lime')
bars2 = ax.bar(x, [np.mean(rmse_values[group]['GRU_CUBIC_PR-AM']) for group in labels], width, label='GRU - CUBIC', color='green')
bars3 = ax.bar(x + width, [np.mean(rmse_values[group]['LSTM_BBR_PR-AM']) for group in labels], width, label='LSTM - BBR', color='salmon')
bars4 = ax.bar(x + 2*width, [np.mean(rmse_values[group]['LSTM_CUBIC_PR-AM']) for group in labels], width, label='LSTM - CUBIC', color='firebrick')

# Configurações do gráfico
ax.set_xlabel('Técnicas de Imputação', size= 24)
ax.set_ylabel('RMSE', size = 22)
# ax.set_title('RMSE - Ponto de Comunicação PR-AM', size = 24)
ax.set_xticks(x)
ax.set_xticklabels(labels, size = 20)
ax.legend()
ax.legend(fontsize = 17)


# plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()

### Future Work

In [ ]:
#carregando modelo salvo
caminho = r'..\..\modelo_salvo\interpolacao-ponder-no-tempo - preenchido bbr esmond data pa-ba 06-10-2023.csv - LSTM - final_model.keras'
model_g = load_model(caminho)
prediction_teste = prediction(model_g, X_test, y_test, scaler, 'GRU', substring_desejada)
gru_evaluation = evaluate_prediction(prediction_teste, y_test, 'GRU')

In [ ]:
# automatizando para plotar o grafico de predição usando o arquivo de modelo salvo - em andamento 
import os

# Defina o diretório raiz onde deseja iniciar a busca
diretorio_raiz = r'..\..\modelo_salvo'

# Itere pelos diretórios e subdiretórios
for pasta_raiz, subpastas, arquivos in os.walk(diretorio_raiz):
    for arquivo in arquivos:
        # Verifique se o arquivo é um arquivo Keras
        if arquivo.endswith('.keras'):
            caminho_arquivo = os.path.join(pasta_raiz, arquivo)
            
            # Dividir o nome do arquivo em partes
            partes_nome = arquivo.split(' - ')
            
            if len(partes_nome) >= 3:
                # Extrair a primeira parte e dividi-la por espaços
                primeira_parte = partes_nome[0].split()  # ["knn", "-", "preenchido", "cubic", "esmond", "data", "pr-am", "06-10-2023.csv"]
                
                if len(primeira_parte) >= 7:
                    knn = primeira_parte[0]  # "knn"
                    preenchido = primeira_parte[3]  # "cubic"
                    pa_ba = primeira_parte[6]  # "pr-am"
                    
                    # Extrair a segunda parte diretamente
                    lstm = partes_nome[1].strip()  # "LSTM"
                    
                    print(f"knn: {knn}, preenchido: {preenchido}, pa-ba: {pa_ba}, LSTM: {lstm}")
                else:
                    print(f"Formato inesperado na primeira parte do arquivo: {arquivo}")
            else:
                print(f"Formato inesperado no nome do arquivo: {arquivo}")


In [ ]:
# Future Work - Expanding the Model
#script para pegar os valores do arquivo de saida evaluation_rmse_mae_2.json e gerar o arquivo em json formatado no formato seguinte: 
# {
#     "Interpolação Linear": [
#       {"RMSE": 48.55109337001047, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"}
#     ],
#     "KNN": [
#       {"RMSE": 58.34572597130633, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"}
#     ],
#     "Média Móvel": [
#       {"RMSE": 261.79090368975835, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"}
#     ],
#     "Mediana Móvel": [
#       {"RMSE": 55.576985085313545, "MODELO": "GRU", "TCP": "BBR", "LINK": "PA-BA"}
#     ]
# } - arquivo de saida

import json

# Caminho do arquivo JSON
caminho_arquivo = "../../graficos/predicoes/round_2/evaluation_rmse_mae_2.json"
arquivo_saida = "../../graficos/predicoes/round_2/evaluation_rmse_mae_2_mediamovelpram.json"
try:
        # Abrir o arquivo JSON em modo de leitura
    with open(caminho_arquivo, 'r') as arquivo:
        # Carregar o conteúdo do arquivo JSON para um dicionário
        dados = json.load(arquivo)
        dicionario = {}
        lista = []
    #extraindo informações de modelo, protocolo, link e metrica do arquivo:
        keys = []
        for k, v in dados.items():
            try: 
                k_list = k.split(" ")
                protocol = k_list[3]
                model = k_list[-1]
                link = k_list[6]
                metrica = k_list[0] 
                rmse = v.get("RMSE")
                mae = v.get("MAE")
                nrmse = v.get("NRMSE")
#dados de vazao que não estão com o formato esperado e dá o erro 'list index out of range'
            except IndexError:
                continue
            # Verificar se a chave já existe no dicionário
            if metrica in dicionario:
                # Adicionar os novos valores à lista associada à chave existente
                dicionario[metrica].append({
                    "RMSE": v["RMSE"],
                    "MAE" : v["MAE"],
                    "NRMSE": v["NRMSE"],
                    "MODELO": model,
                    "TCP": protocol.upper(),
                    "LINK": link.upper()
                })
            else:
                # Se a chave não existir, criar uma nova entrada no dicionário
                dicionario[metrica] = [{
                    "RMSE": v["RMSE"],
                    "MAE" : v["MAE"],
                    "NRMSE": v["NRMSE"],
                    "MODELO": model,
                    "TCP": protocol.upper(),
                    "LINK": link.upper()
                }]
#alterando as chavesa do dicionarios, para que ao plotar o grafico eles ja fiquem com os nomes corretos
#exemplo: ao inves de ficar media-movel na label do grafico, fica Media Movel.
        #dicionario['Media Movel'] = dicionario.pop('media-movel') 

        dicionario['Moving Average'] = dicionario.pop('media-movel') 
        #dicionario['Interpolacao Linear'] = dicionario.pop('interpolacao-linear')
        dicionario['Linear Interpolation'] = dicionario.pop('interpolacao-linear')
        #dicionario['Interpolacao Ponderada no Tempo'] = dicionario.pop('interpolacao-ponder-no-tempo')
        # dicionario['Time-weighted Averages'] = dicionario.pop('interpolacao-ponder-no-tempo')
        dicionario['KNN'] = dicionario.pop('knn')
        #dicionario['Media Dia Repetido'] = dicionario.pop('media-dia-repetido')
        # dicionario['Mean of Repeated Measurements'] = dicionario.pop('media-dia-repetido')
        #dicionario['Mediana Movel'] = dicionario.pop('mediana-movel')
        dicionario['Moving Median'] = dicionario.pop('mediana-movel')
        #dicionario['Multi Time'] = dicionario.pop('multi-time')
        # dicionario['Multiple Imputation'] = dicionario.pop('multi-time')
        dicionario['SVD'] = dicionario.pop('svd')
        with open('../../graficos/predicoes/round_2/evaluation_rmse_mae_2_formatado.json', 'w') as arquivo_saida:
                json.dump(dicionario, arquivo_saida, indent=4)
    

except FileNotFoundError:
    print("Arquivo não encontrado. Certifique-se de que o caminho está correto e tente novamente.")
except json.JSONDecodeError:
    print("Erro ao decodificar o arquivo JSON. Verifique se o arquivo contém um JSON válido.")
except Exception as e:
    print("Ocorreu um erro:", e)

In [ ]:
# Future Work - Expanding the Model
#plotando dados de RMSE

import matplotlib.pyplot as plt
import numpy as np

arquivo = "../../graficos/predicoes/round_2/evaluation_rmse_mae_2_formatado.json"
    # Abrir o arquivo JSON em modo de leitura
with open(arquivo, 'r') as arquivo:
    # Carregar o conteúdo do arquivo JSON para um dicionário
    data = json.load(arquivo)

rmse_values = {
    'interpolacao-linear': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'knn': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'media-movel': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'mediana-movel': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []}, 
    'multi-time': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'interpolacao-ponderada-no-tempo': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'media-dia-repetido': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []}

}

for group, values in data.items():
    for item in values:
        rmse = item['RMSE']
        modelo = item['MODELO']
        tcp = item['TCP']
        link = item['LINK']
        
        if modelo == 'GRU' and tcp == 'BBR' and link == 'PR-AM':
            rmse_values[group]['GRU_BBR_PR-AM'].append(rmse)
        elif modelo == 'LSTM' and tcp == 'CUBIC' and link == 'PR-AM':
            rmse_values[group]['LSTM_CUBIC_PR-AM'].append(rmse)
        elif modelo == 'LSTM' and tcp == 'BBR' and link == 'PR-AM':
            rmse_values[group]['LSTM_BBR_PR-AM'].append(rmse)
        elif modelo == 'GRU' and tcp == 'CUBIC' and link == 'PR-AM':
            rmse_values[group]['GRU_CUBIC_PR-AM'].append(rmse)
        else:
            #depois tem que informar os outros casos
            pass



# Criar o gráfico de barras
labels = list(rmse_values.keys())
x = np.arange(len(labels))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 6))

# Adicionar as barras
bars1 = ax.bar(x - width, [np.mean(rmse_values[group]['GRU_BBR_PR-AM']) for group in labels], width, label='GRU - BBR', color='lime')
bars2 = ax.bar(x, [np.mean(rmse_values[group]['GRU_CUBIC_PR-AM']) for group in labels], width, label='GRU - CUBIC', color='green')
bars3 = ax.bar(x + width, [np.mean(rmse_values[group]['LSTM_BBR_PR-AM']) for group in labels], width, label='LSTM - BBR', color='salmon')
bars4 = ax.bar(x + 2*width, [np.mean(rmse_values[group]['LSTM_CUBIC_PR-AM']) for group in labels], width, label='LSTM - CUBIC', color='firebrick')


# Configurações do gráfico
ax.set_xlabel('Técnicas de Imputação', size= 12)
ax.set_ylabel('RMSE', size = 12)
# ax.set_title('RMSE - Ponto de Comunicação PR-AM', size = 24)
ax.set_xticks(x)
ax.set_xticklabels(labels, size = 8)
ax.legend()
ax.legend(fontsize = 12)


# plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()

In [ ]:
# Future Work - Expanding the Model

#plotando dados de MAE
import matplotlib.pyplot as plt
import numpy as np

arquivo = "../../graficos/predicoes/round_2/evaluation_rmse_mae_2_formatado.json"
    # Abrir o arquivo JSON em modo de leitura
with open(arquivo, 'r') as arquivo:
    # Carregar o conteúdo do arquivo JSON para um dicionário
    data = json.load(arquivo)

mae_values = {
    'interpolacao-linear': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'knn': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'media-movel': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'mediana-movel': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []}, 
    'multi-time': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'interpolacao-ponderada-no-tempo': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []},
    'media-dia-repetido': {'GRU_BBR_PR-AM': [], 'LSTM_CUBIC_PR-AM': [], 'LSTM_BBR_PR-AM': [], 'GRU_CUBIC_PR-AM': []}

}

for group, values in data.items():
    for item in values:
        mae = item['MAE']
        modelo = item['MODELO']
        tcp = item['TCP']
        link = item['LINK']
        
        if modelo == 'GRU' and tcp == 'BBR' and link == 'PR-AM':
            mae_values[group]['GRU_BBR_PR-AM'].append(mae)
        elif modelo == 'LSTM' and tcp == 'CUBIC' and link == 'PR-AM':
            mae_values[group]['LSTM_CUBIC_PR-AM'].append(mae)
        elif modelo == 'LSTM' and tcp == 'BBR' and link == 'PR-AM':
            mae_values[group]['LSTM_BBR_PR-AM'].append(mae)
        elif modelo == 'GRU' and tcp == 'CUBIC' and link == 'PR-AM':
            mae_values[group]['GRU_CUBIC_PR-AM'].append(mae)
        else:
            pass


# Criar o gráfico de barras
labels = list(mae_values.keys())
x = np.arange(len(labels))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 6))

# Adicionar as barras
bars1 = ax.bar(x - width, [np.mean(mae_values[group]['GRU_BBR_PR-AM']) for group in labels], width, label='GRU - BBR', color='lime')
bars2 = ax.bar(x, [np.mean(mae_values[group]['GRU_CUBIC_PR-AM']) for group in labels], width, label='GRU - CUBIC', color='green')
bars3 = ax.bar(x + width, [np.mean(mae_values[group]['LSTM_BBR_PR-AM']) for group in labels], width, label='LSTM - BBR', color='salmon')
bars4 = ax.bar(x + 2*width, [np.mean(mae_values[group]['LSTM_CUBIC_PR-AM']) for group in labels], width, label='LSTM - CUBIC', color='firebrick')


# Configurações do gráfico
ax.set_xlabel('Técnicas de Imputação', size= 12)
ax.set_ylabel('MAE', size = 12)
# ax.set_title('RMSE - Ponto de Comunicação PR-AM', size = 24)
ax.set_xticks(x)
ax.set_xticklabels(labels, size = 8)
ax.legend()
ax.legend(fontsize = 12)


# plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()

In [ ]:
# Preliminary tests (ignore)
import json
import os

# ajuste texto dos plots
for i in range (1,6):
    evaluation_round = {}
    path = '../../graficos/predicoes/round_{}/evaluation_rmse_mae_{}.json'.format(i,i)
    
    parts = path.split("/")
    round = parts[4] # pegar o round de treinamento
    
    print(round)
    # with open(path, 'r') as arquivo:
    #     evaluation_round = json.load(arquivo)
    #     chaves = list(evaluation_round.keys())

    #     for i in range(len(chaves)):
    #         chaves[i] = chaves[i].replace('preenchido',"")
    #         chaves[i] = chaves[i].replace('-repetido - ',"")
    #         chaves[i] = chaves[i].replace('media_riginal.csv',"")
    #         chaves[i] = chaves[i].replace('esmond data ',"")
    #         chaves[i] = chaves[i].replace('06-10-2023.csv, ',"")
    #         chaves[i] = chaves[i].replace('06-10-2023_',"")
    #         chaves[i] = chaves[i].replace('07-08-2023.csv, ',"")
    #         chaves[i] = chaves[i].replace('-tratado\\intervalos vazao',"")
    #         chaves[i] = chaves[i].replace('interpolacao-linear\\',"interpolacao-linear")
    #         chaves[i] = chaves[i].replace('interpolacao-ponderada-no-tempo\\',"interpolacao-ponderada-no-tempo")
    #         # chaves[i] = chaves[i].replace('media-movel',"MA")
    #         # chaves[i] = chaves[i].replace('mediana-movel',"MM")
    #         chaves[i] = chaves[i].replace('knn\\',"knn")
    #         chaves[i] = chaves[i].replace('media-movel\\',"media-movel")
    #         chaves[i] = chaves[i].replace('mediana-movel\\',"mediana-movel")


    # valores_mae = [item['MAE'] for item in evaluation_round.values()]
    # valores_rmse = [item['RMSE'] for item in evaluation_round.values()]

    # cores = {
    #     'interpolacao-linear': 'skyblue',
    #     'interpolacao-ponderada-no-tempo': 'salmon',
    #     'knn': 'lightgreen',
    #     'media-movel': 'green',
    #     'mediana-movel': 'mediumpurple',
    #     'multi-time':'red',
    #     'media-dia': 'gold'
    # }

    # #Extraindo a primeira parte de cada categoria
    # primeiras_partes = [categoria.split(' ', 1)[0] for categoria in chaves]

    # cores_barras = [cores[parte] for parte in primeiras_partes]

    # # for i in range(len(chaves)):
    # #     chaves[i] = chaves[i].upper()


    # # Criar o gráfico de barras
    # plt.figure(figsize=(16, 6))
    # plt.bar(chaves, valores_mae, color=cores_barras)
    # plt.xlabel('Categorias')
    # plt.ylabel('Valores de MAE')
    # plt.title('Valores de MAE Predições Round ' + round)
    # plt.xticks(rotation=90)  # Rotacionar os nomes das categorias no eixo x para melhor legibilidade
    # plt.ylim(0, 400)

    # plt.show()

    # # Criar o gráfico de barras
    # plt.figure(figsize=(16, 6))
    # plt.bar(chaves, valores_rmse, color=cores_barras)
    # plt.xlabel('Categorias')
    # plt.ylabel('Valores de RMSE')
    # plt.title('Valores de RMSE Predições Round ' + round)
    # plt.xticks(rotation=90)  # Rotacionar os nomes das categorias no eixo x para melhor legibilidade
    # plt.ylim(0, 400)
    # plt.show()

In [ ]:
# em progresso: 
def grid_search_cv(modelo, units, X_train, learning_rates, y_train, epochs_list, batch_sizes, patiences, model_name):
    best_loss = float('inf')
    best_params = {}
    best_model = None  
    
    for lr in learning_rates:
        for epochs in epochs_list:
            for batch_size in batch_sizes:
                for patience in patiences:
                    model = modelo(units, X_train, lr)
                    histories = fit_model_with_cross_validation(model, X_train, y_train, model_name, patience, epochs, batch_size)
                    mean_history = calculate_mean_history(histories)
                    val_loss = min(mean_history['val_loss'])
                    print("Val Loss: ", val_loss, "learning rate: ", lr, "epochs: ", epochs, "batch_size: ", batch_size, "patience: ", patience)
                    if val_loss < best_loss:
                        best_loss = val_loss
                        best_params = {'learning_rate': lr, 'epochs': epochs, 'batch_size': batch_size, 'patience': patience}
                        best_model = model  
    print('O modelo ' + model_name + ' tem como melhores parâmetros os seguintes: learning_rate ' + 
          str(best_params['learning_rate']) + ' epochs: ' + str(best_params['epochs']) +
          ' batch_size: ' + str(best_params['batch_size']) + ' patience: ' + str(best_params['patience']))
    
    return best_model, best_params 
